> **SOLUTION VERSION.** Every exercise is completed. Stuck in the workshop? Copy what you need from here.

# Lab 2: Giving the Model Tools
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmdayal/from-prompt-to-agent/blob/main/solutions/02_tools.ipynb)

An LLM cannot run code, browse your files or read your database. But it **can** write a request,
like the LLM volunteer in the role-play who wrote on a card: *"Tool, please search for agents sessions."*

The pattern:
1. We give the model a **menu of tools** (name + what it does)
2. The model **chooses** a tool and writes the arguments as JSON
3. **Our code runs the tool** (the model never runs anything itself)
4. We hand the **result** back to the model so it can answer

### Setup
Run the cell below. The last line tells you which **brain** is answering:
`openai` or `gemini` (real AI in the cloud), `ollama` (AI on your own computer) or `mock`
(offline practice brain). All of them work for every exercise in this workshop.

In Colab, the first time a notebook reads your key, a pop-up asks for permission:
click **Grant access**. Every lab asks once.

In [ ]:
# SETUP: run this cell first. It is safe to run more than once.
import os, sys, subprocess
REPO_URL = "https://github.com/bmdayal/from-prompt-to-agent"

if os.path.isdir("../workshop"):        # running locally from notebooks/ or solutions/
    sys.path.insert(0, "..")
elif os.path.isdir("workshop"):         # running from the repo folder
    sys.path.insert(0, ".")
else:                                   # running in Google Colab: download the workshop files
    if not os.path.isdir("from-prompt-to-agent"):
        print("Downloading workshop files...")
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    sys.path.insert(0, "from-prompt-to-agent")

import json
from workshop import llm

PROVIDER = "auto"   # "auto", "openai", "gemini", "ollama" or "mock" (offline practice brain)
llm.configure(PROVIDER)

## Step 1: A tool is just a Python function
Nothing magic here. These are ordinary functions that search our program data.

In [ ]:
from workshop.tools import SESSIONS

def search_sessions(topic="", day=""):
    """Find sessions whose title, tags or track mention a topic, optionally on one day."""
    results = []
    for s in SESSIONS:
        text = " ".join([s["title"], s["track"], " ".join(s["tags"])]).lower()
        if topic and topic.lower() not in text:
            continue
        if day and s["day"].lower() != day.lower():
            continue
        results.append({k: s[k] for k in ("id", "title", "day", "start", "end", "room")})
    return results

def get_session_details(session_id):
    """Return everything about one session, for example S04."""
    for s in SESSIONS:
        if s["id"] == session_id.upper():
            return s
    return {"error": f"No session with id {session_id}"}

# Try the tools ourselves first. No AI involved yet!
print(search_sessions("security"))
print(get_session_details("S04")["speaker"])

## Step 2: The menu and the rules
The model only knows about a tool if we **describe** it. Good descriptions lead to good choices.

In [ ]:
TOOLS = {
    "search_sessions": search_sessions,
    "get_session_details": get_session_details,
}

TOOL_MENU = """- search_sessions(topic, day): find sessions about a topic (e.g. "agents", "security") on an optional day ("Thursday", "Friday", "Saturday"). Either argument may be "".
- get_session_details(session_id): full details (speaker, room, time, description) for one session id like S04."""

def build_system(menu):
    return f"""You are the iGET Conference Concierge. You can use these tools:
{menu}

Reply with ONE JSON object and nothing else:
{{"thought": "your short reasoning", "action": "<tool name or final_answer>", "args": {{...}}}}
When you know the answer, use "action": "final_answer" with "args": {{"answer": "..."}}.
Only state facts that came from tool results."""

SYSTEM = build_system(TOOL_MENU)
print(SYSTEM)

## Step 3: Let the model choose
**Predict:** which tool will it pick for this question, and with which arguments?

In [ ]:
question = "When is the session on prompt engineering?"

reply = llm.ask(question, system=SYSTEM, json_mode=True)
decision = llm.parse_json(reply)
print("The model decided:")
print(json.dumps(decision, indent=2))

## Step 4: Our code runs the tool
The model only **asked**. Now our program actually does the work.

In [ ]:
if decision and decision.get("action") in TOOLS:
    tool = TOOLS[decision["action"]]
    result = tool(**decision.get("args", {}))
    print("Tool result:")
    print(json.dumps(result, indent=2))
else:
    result = {"error": "the model did not choose a valid tool"}
    print("The model did not pick a tool this time:", decision)

## Step 5: Give the result back
We add the model's request and the tool's result (an **Observation**) to the conversation,
then ask again.

In [ ]:
messages = [
    {"role": "user", "content": question},
    {"role": "assistant", "content": reply},
    {"role": "user", "content": "Observation: " + json.dumps(result)},
]
decision2 = llm.parse_json(llm.ask(messages, system=SYSTEM, json_mode=True))

if decision2 and decision2.get("action") == "final_answer":
    print("FINAL ANSWER:", decision2["args"]["answer"])
else:
    print("The model wants ANOTHER tool:", json.dumps(decision2))
    print("Doing this by hand gets tedious... in Lab 3 a loop will do it for us.")

## Your turn: write a new tool
Write `find_by_speaker(name)`, which returns the sessions whose speaker name contains `name`.

*Hint:* loop over `SESSIONS` and check `name.lower() in s["speaker"].lower()`.

In [ ]:
def find_by_speaker(name):
    """Find sessions given by a speaker (full or partial name)."""
    return [{"id": s["id"], "title": s["title"], "day": s["day"], "start": s["start"],
             "room": s["room"], "speaker": s["speaker"]}
            for s in SESSIONS if name.lower() in s["speaker"].lower()]

print(find_by_speaker("Martinez"))

Now add it to the menu so the model knows it exists, and test it.

In [ ]:
TOOLS["find_by_speaker"] = find_by_speaker
SYSTEM2 = build_system(TOOL_MENU + "\n- find_by_speaker(name): find sessions given by a speaker (full or partial name).")

question = "What is Sofia Martinez presenting?"
decision = llm.parse_json(llm.ask(question, system=SYSTEM2, json_mode=True))
print("The model decided:", decision)
if decision and decision.get("action") in TOOLS:
    print("Tool result:", TOOLS[decision["action"]](**decision["args"]))

**Stretch goal:** write `sessions_in_room(room)` and add it to the menu.

## Key takeaways
- The model **chooses**, your code **executes**. You stay in control of what is possible.
- Tool **descriptions** are prompts too. Vague descriptions cause wrong choices.
- Each step is still structured output (JSON) from Lab 1.

**Next: Lab 3, The Agent Loop and Memory.** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmdayal/from-prompt-to-agent/blob/main/solutions/03_agent_loop_and_memory.ipynb)

Click the badge to open the next lab in a new Colab tab, then run its setup cell first. (Each lab is self-contained, so you can close this one.)